# E-14 Document AI & ETL Pipeline | Fase 1: Ingestión, Muestreo Estratificado y Scraping Concurrente

Notebook demostrativo de la **Fase 1** del pipeline:
1. **Ingestión paginada** del catálogo maestro DIVIPOLE (`divipole_regis`) desde Supabase (PostgreSQL).
2. **Delimitación del universo doméstico** (exclusión de consulados `dd = '88'`).
3. **Diseño experimental probabilístico** mediante la fórmula de Cochran para poblaciones finitas ($95\%$ confianza, $\pm 5\%$ margen de error) con estratificación proporcional (`urbano`, `rural`, `carcel`).
4. **Muestreo atómico reproducible** a nivel de mesa (`SEED=42`) y **scraping concurrente dirigido** de los formularios PDF E-14.

## 1. Instalación de Dependencias (Solo Google Colab)

In [1]:
import sys

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q supabase python-dotenv pandas numpy curl_cffi

## 2. Configuración del Entorno, Módulos (`src/`) y Conexión a Supabase

In [2]:

import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
from supabase import create_client

# Ajuste automático de la raíz del proyecto según el entorno de ejecución
PROJECT_ROOT = (
    Path("/content/drive/MyDrive/Elecciones2_2026_PDFs/e14")
    if IS_COLAB
    else Path.cwd().parent
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.ingestion.extract_e14_sample import (
    fetch_all_rows,
    get_sample_size,
    get_stratum,
    sample_voting_tables,
)
from src.scraper.scraper_e14 import execute_scraper

# Carga segura de credenciales desde el archivo .env en la raíz del repositorio
#   SUPABASE_URL= "https://xyzxyzxyzyxz.supabase.co"
#   SUPABASE_KEY= key_de_supabase

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("Faltan SUPABASE_URL o SUPABASE_KEY en el archivo .env de la raíz.")


# Crea la conexion entre la sesion de Colab y Supabase
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

## 3. Extracción Paginada del Catálogo DIVIPOLE desde Supabase

In [3]:
TABLE_NAME = "divipole_regis"
COLUMNS = "dd,mm,zz,pp,departamento,municipio,puesto,comuna,mesas_domingo"

rows = fetch_all_rows(supabase, TABLE_NAME, COLUMNS)
df = pd.DataFrame(rows)

print(f"Puestos extraídos de Supabase: {len(df):,}")
df.head()

Puestos extraídos de Supabase: 13,742


,dd,mm,zz,pp,departamento,municipio,puesto,comuna,mesas_domingo
0,01,001,01,01,ANTIOQUIA,MEDELLIN,SEC. ESC. LA ESPERANZA No 2,01COMUNA 1\nPOPULAR,36
1,01,001,01,02,ANTIOQUIA,MEDELLIN,INST.EDUC. LA CANDELARIA,01COMUNA 1\nPOPULAR,44
2,01,001,01,03,ANTIOQUIA,MEDELLIN,IE MARIA DE LOS ANGELES CANO MARQUEZ,01COMUNA 1\nPOPULAR,25
3,01,001,01,04,ANTIOQUIA,MEDELLIN,SEC. ESC. MEDELLIN,01COMUNA 1\nPOPULAR,26
4,01,001,01,05,ANTIOQUIA,MEDELLIN,I.E.FE Y ALEGRIA GRANIZAL,01COMUNA 1\nPOPULAR,15


## 4. Delimitación del Universo Estadístico (Exclusión de Consulados)

Se excluyen las mesas consulares (`dd == '88'`) por ausencia de actas escaneadas en el portal de auditoría nacional, consolidando el universo doméstico evaluable.

In [4]:
df = df[df["dd"] != "88"].copy()

print(f"Puestos tras excluir consulados (dd=88): {len(df):,}")
print(f"Total mesas en universo doméstico (N)  : {int(df['mesas_domingo'].sum()):,}")

Puestos tras excluir consulados (dd=88): 13,489
Total mesas en universo doméstico (N)  : 118,346


## 5. Estratificación por Zona (`zz`) y Cálculo Muestral (Cochran)

Reglas oficiales de zonificación DIVIPOLE:
* **`zz = 99`** $\rightarrow$ `rural` (veredas y corregimientos).
* **`zz = 98`** $\rightarrow$ `carcel` (puestos penitenciarios).
* **`zz = 00` a `97`** $\rightarrow$ `urbano` (cabeceras municipales y comunas).

In [6]:
# Clasificación por estrato operativo y cálculo de tamaño muestral con Cochran
df["estrato"] = df["zz"].apply(get_stratum)
N = int(df["mesas_domingo"].sum())

CONFIDENCE = 0.95
MARGIN_ERROR = 0.05
n = get_sample_size(N, confidence=CONFIDENCE, margin=MARGIN_ERROR)

# Asignación proporcional de cuotas con corrección del residuo por redondeo
stratum_counts = df.groupby("estrato")["mesas_domingo"].sum()
stratum_pct = (stratum_counts / N * 100).round(2)
stratum_allocation = (stratum_counts / N * n).round().astype(int)

rounding_diff = n - int(stratum_allocation.sum())
if rounding_diff != 0:
    stratum_allocation[stratum_counts.idxmax()] += rounding_diff

print(f"Universo total (N): {N:,} mesas | Muestra requerida ({int(CONFIDENCE * 100)}% / ±{int(MARGIN_ERROR * 100)}%): n = {n}")

summary_df = pd.DataFrame(
    {
        "Total_Mesas": stratum_counts,
        "Participacion_%": stratum_pct,
        "Cuota_Muestra": stratum_allocation,
    }
)
summary_df

Universo total (N): 118,346 mesas | Muestra requerida (95% / ±5%): n = 383


,Total_Mesas,Participacion_%,Cuota_Muestra
estrato,,,
carcel,162,0.14,1
rural,19365,16.36,63
urbano,98819,83.50,319


## 6. Muestreo Aleatorio Simple a Nivel de Mesa y Exportación CSV


In [7]:
# Expansión a nivel mesa individual y muestreo reproducible (SEED=42)
sample_df = sample_voting_tables(df, stratum_allocation)

output_csv = PROJECT_ROOT / "data" / "muestra_e14_segunda_vuelta.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
sample_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Total mesas muestreadas: {len(sample_df)} | Guardado en: {output_csv}")
print(sample_df["estrato"].value_counts())
sample_df.head()

Total mesas muestreadas: 383 | Guardado en: g:\Mi unidad\Elecciones2_2026_PDFs\e14\data\muestra_e14_segunda_vuelta.csv
estrato
urbano    319
rural      63
carcel      1
Name: count, dtype: int64


,dd,mm,zz,pp,departamento,municipio,puesto,mesa,estrato
0,64,001,98,01,PUTUMAYO,MOCOA,CARCEL - CASA DE JUSTICIA,1,carcel
1,64,001,99,41,PUTUMAYO,MOCOA,VEREDA PUEBLO VIEJO,2,rural
2,05,059,99,31,BOLIVAR,PINILLOS,PUERTO LOPEZ,3,rural
3,05,022,99,17,BOLIVAR,EL CARMEN DE BOLIVAR,MACAYEPO,4,rural
4,13,010,99,13,CORDOBA,CERETE,I.E JULIAN PINTO BUENDIA BTO MATEO GOMEZ,12,rural


## 7. Scraping Concurrente y Descarga Dirigida de Actas PDF E-14

Resuelve el árbol JSON oficial (`index.json` $\rightarrow$ `divipole.json` $\rightarrow$ JSON por puesto) únicamente para los puestos muestreados y descarga los PDFs validando su cabecera binaria `%PDF`.

In [8]:
execute_scraper()

[Selección] 367 puesto(s) único(s) | 383 mesa(s) a descargar

[=== FASE 1: DESCARGA DE ESTRUCTURAS MAESTRAS ===]
  [Caché] Leyendo localmente: index.json
  [Caché] Leyendo localmente: divipole.json

[=== FASE 2: RESOLUCIÓN DE RUTAS (SOLO PUESTOS SELECCIONADOS) ===]
[*] Puesto seleccionado: 01/001/02/02 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_02_02_mesas_20260621_210742_815.json
[*] Puesto seleccionado: 01/001/06/02 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_06_02_mesas_20260621_192741_771.json
[*] Puesto seleccionado: 01/001/09/05 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_09_05_mesas_20260621_193752_029.json
[*] Puesto seleccionado: 01/001/10/03 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_10_03_mesas_20260621_185606_874.json
[*] Puesto seleccionado: 01/001/11/04 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_11_04_mesas_20260621_183245_345.json
[*] Puest